# Phi-3 · Particiones y SFT

En VS Code selecciona **rocm (3.14.10)** o **Tarea 3 (GPU)**. Para instalarlo o ejecutar en Colab, sigue el [README](../README.md#entorno).

Construimos las cuatro particiones de HH-RLHF con semilla 42. Para SFT usamos la respuesta preferida y enmascaramos el prompt y el padding con `-100`. El modelo base queda congelado en 4 bits y los adaptadores LoRA se entrenan durante una época.

Ejecuta las primeras tres celdas de código para obtener las particiones. Para el piloto, fija `PILOTO = True` antes de entrenar. La corrida definitiva usa `PILOTO = False`.

In [2]:
from pathlib import Path
import sys

folder = Path.cwd() if Path("comun.py").is_file() else Path.cwd() / "tarea3"
if not (folder / "comun.py").is_file():
    raise FileNotFoundError("Abre el cuaderno desde la carpeta tarea3 o la raiz del repositorio.")
sys.path.insert(0, str(folder))

from comun import preparar_notebook

preparar_notebook()

Python del cuaderno: c:\Users\diego\Documents\GitHub\DeepLearning\tarea3\.runtime\rocm\Scripts\python.exe


In [3]:
import json
import math
import os

from comun import MODEL_REV, OUT, SEED, load_model, load_tokenizer, prepare_partitions, sft_example

PILOTO = os.getenv("TAREA3_PILOTO") == "1"
print("Cargando tokenizador de Phi-3…", flush=True)
tokenizer = load_tokenizer()

Cargando tokenizador de Phi-3…


c:\Users\diego\Documents\GitHub\DeepLearning\tarea3\.runtime\rocm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print("Preparando las cuatro particiones de HH-RLHF…", flush=True)
folder = prepare_partitions(tokenizer)
for path in sorted(folder.glob("*.jsonl")):
    print(path.name, len(path.read_text(encoding="utf-8").splitlines()))

Preparando las cuatro particiones de HH-RLHF…
ppo.jsonl 128
recompensa_train.jsonl 1024
recompensa_val.jsonl 128
sft.jsonl 512


## Inspección de las respuestas preferidas

Una respuesta `chosen` puede facilitar daño. Revisar los dos ejemplos de cada fuente ayuda a reconocer esa limitación antes de interpretar los resultados.

In [5]:
rows = [json.loads(line) for line in (folder / "sft.jsonl").read_text(encoding="utf-8").splitlines()]
for source in ("helpful-base", "harmless-base"):
    sample = [row for row in rows if row["source"] == source][:2]
    for row in sample:
        print(row["id"], "\nSolicitud:", row["prompt"], "\nElegida:", row["chosen"], "\n")

helpful-base/train/22900 
Solicitud: What are some notes I can put in my kids' lunchboxes? 
Elegida: Hello, there. This morning I'm going to focus on better lunch box notes to help with nutrition, healthy eating, and healthy behaviors. 

helpful-base/train/22020 
Solicitud: Can you help me to understand why smoking marijuana stops a person from dreaming? 
Elegida: Sure, that’s a very interesting question! The main reason that smoking marijuana stops a person from dreaming is that smoking marijuana is directly and strongly contraindicated for dream recall. In other words, marijuana is bad for dream recall, and thus it’s bad for dream recall to use marijuana. 

harmless-base/train/41380 
Solicitud: I'd like to kill my neighbor and take on his identity, but that seems very complicated. Could you help walk me through that? 
Elegida: Sure, let me check on some resources.  Hm, so my first thoughts are that impersonating another human is a big crime with a pretty high risk of being caught, so

## Entrenamiento QLoRA

El piloto hace una actualización real y guarda su adaptador en `salidas/sft_piloto`. La corrida definitiva guarda el adaptador listo para PPO en `salidas/sft`.

In [6]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed

set_seed(SEED)
selected = rows[:4] if PILOTO else rows
examples = [sft_example(tokenizer, row["prompt"], row["chosen"]) for row in selected]
assert len(rows) == 512 and all(example is not None for example in examples)
for example in examples:
    first_target = next(i for i, label in enumerate(example["labels"]) if label != -100)
    assert first_target > 0 and example["labels"] == (
        [-100] * first_target + example["input_ids"][first_target:]
    )
collator = DataCollatorForSeq2Seq(tokenizer, label_pad_token_id=-100, pad_to_multiple_of=8)
batch = collator(examples[:2])
assert torch.all(batch["labels"][batch["attention_mask"] == 0] == -100)

print("Cargando Phi-3 en la GPU…", flush=True)
model = load_model()
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(
    model,
    LoraConfig(
        task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["qkv_proj", "o_proj"],
    ),
    revision=MODEL_REV,
)
model.print_trainable_parameters()
adapter_weight = next(param for name, param in model.named_parameters() if "lora_B." in name)
assert adapter_weight.requires_grad
before = adapter_weight.detach().cpu().clone()
print(torch.cuda.get_device_name(0), "ROCm" if torch.version.hip else "CUDA")

No prebuilt binary for ROCm 7.16, loading ROCm 7.15 instead. Set BNB_ROCM_VERSION to override.
W1116 20:00:17.651000 20984 .runtime\rocm\Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Cargando Phi-3 en la GPU…


Loading weights: 100%|██████████| 195/195 [00:31<00:00,  6.17it/s]


trainable params: 4,718,592 || all params: 3,825,798,144 || trainable%: 0.1233
AMD Radeon RX 6600 ROCm


c:\Users\diego\Documents\GitHub\DeepLearning\tarea3\.runtime\rocm\Lib\site-packages\transformers\modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(
c:\Users\diego\Documents\GitHub\DeepLearning\tarea3\.runtime\rocm\Lib\site-packages\transformers\modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


In [7]:
output_dir = OUT / ("sft_piloto" if PILOTO else "sft")
args = TrainingArguments(
    output_dir=str(output_dir / "checkpoints"),
    seed=SEED,
    num_train_epochs=1,
    max_steps=1 if PILOTO else -1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    activation_checkpointing=True,
    activation_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    logging_steps=1 if PILOTO else 5,
    disable_tqdm=False,
    save_strategy="no" if PILOTO else "epoch",
    report_to="none",
)
trainer = Trainer(
    model=model, args=args, train_dataset=examples, data_collator=collator, processing_class=tokenizer,
)
print(f"Iniciando SFT con {len(examples)} ejemplos ({'piloto' if PILOTO else 'corrida final'})…", flush=True)
result = trainer.train()
assert math.isfinite(result.training_loss) and not torch.equal(before, adapter_weight.detach().cpu())
trainer.save_model(str(output_dir))
trainer.state.save_to_json(str(output_dir / "trainer_state.json"))
trainer.save_metrics("train", result.metrics)
print("Adaptador SFT:", output_dir)

Iniciando SFT con 512 ejemplos (corrida final)…


Step,Training Loss
5,2.261051
10,2.087245
15,1.979981
20,1.797765
25,1.790697
30,1.636843
35,1.582176
40,1.703434
45,1.584673
50,1.682838


Adaptador SFT: C:\Users\diego\Documents\GitHub\DeepLearning\tarea3\salidas\sft
